### Establishing connection

In [1]:
import duckdb

conn = duckdb.connect(database='../taxi_rides_ny/taxi_rides_ny.duckdb')

In [97]:
tables = conn.execute("""
SELECT table_name, table_type
FROM information_schema.tables
WHERE table_schema = 'dev';
""").fetchall()

print("Tables/Views in 'dev' schema:")
for table_name, table_type in tables:
    print(f"{table_name} ({table_type})")

Tables/Views in 'dev' schema:
dim_vendors (BASE TABLE)
dim_zones (BASE TABLE)
fct_monthly_zone_revenue (BASE TABLE)
fct_trips (BASE TABLE)
fct_trips__dbt_backup (BASE TABLE)
int_trips (BASE TABLE)
int_trips_unioned (BASE TABLE)
payment_type_lookup (BASE TABLE)
taxi_zone_lookup (BASE TABLE)
stg_fhv_tripdata (VIEW)
stg_green_tripdata (VIEW)
stg_yellow_tripdata (VIEW)


### Question 3. Counting Records in `fct_monthly_zone_revenue`

### Alternativa: Query direta nos Parquet (sem dbt build)

In [ ]:
# Query que simula fct_monthly_zone_revenue diretamente dos arquivos Parquet
# Sem criar tabelas - economiza espaço!

result = conn.execute("""
WITH yellow_trips AS (
    SELECT
        z.Zone as pickup_zone,
        DATE_TRUNC('month', y.tpep_pickup_datetime) as revenue_month,
        'Yellow' as service_type,
        y.fare_amount,
        y.extra,
        y.mta_tax,
        y.tip_amount,
        y.tolls_amount,
        0.0 as ehail_fee,
        y.improvement_surcharge,
        y.total_amount,
        y.passenger_count,
        y.trip_distance
    FROM read_parquet('../taxi_rides_ny/data/yellow/yellow_tripdata_2019-*.parquet') y
    LEFT JOIN read_csv('../taxi_rides_ny/seeds/taxi_zone_lookup.csv') z
        ON y.PULocationID = z.LocationID
    WHERE y.tpep_pickup_datetime >= '2019-01-01' 
      AND y.tpep_pickup_datetime < '2021-01-01'
    
    UNION ALL
    
    SELECT
        z.Zone as pickup_zone,
        DATE_TRUNC('month', y.tpep_pickup_datetime) as revenue_month,
        'Yellow' as service_type,
        y.fare_amount,
        y.extra,
        y.mta_tax,
        y.tip_amount,
        y.tolls_amount,
        0.0 as ehail_fee,
        y.improvement_surcharge,
        y.total_amount,
        y.passenger_count,
        y.trip_distance
    FROM read_parquet('../taxi_rides_ny/data/yellow/yellow_tripdata_2020-*.parquet') y
    LEFT JOIN read_csv('../taxi_rides_ny/seeds/taxi_zone_lookup.csv') z
        ON y.PULocationID = z.LocationID
    WHERE y.tpep_pickup_datetime >= '2019-01-01' 
      AND y.tpep_pickup_datetime < '2021-01-01'
),
green_trips AS (
    SELECT
        z.Zone as pickup_zone,
        DATE_TRUNC('month', g.lpep_pickup_datetime) as revenue_month,
        'Green' as service_type,
        g.fare_amount,
        g.extra,
        g.mta_tax,
        g.tip_amount,
        g.tolls_amount,
        CAST(g.ehail_fee AS DOUBLE) as ehail_fee,
        g.improvement_surcharge,
        g.total_amount,
        g.passenger_count,
        g.trip_distance
    FROM read_parquet('../taxi_rides_ny/data/green/green_tripdata_2019-*.parquet') g
    LEFT JOIN read_csv('../taxi_rides_ny/seeds/taxi_zone_lookup.csv') z
        ON g.PULocationID = z.LocationID
    WHERE g.lpep_pickup_datetime >= '2019-01-01' 
      AND g.lpep_pickup_datetime < '2021-01-01'
    
    UNION ALL
    
    SELECT
        z.Zone as pickup_zone,
        DATE_TRUNC('month', g.lpep_pickup_datetime) as revenue_month,
        'Green' as service_type,
        g.fare_amount,
        g.extra,
        g.mta_tax,
        g.tip_amount,
        g.tolls_amount,
        CAST(g.ehail_fee AS DOUBLE) as ehail_fee,
        g.improvement_surcharge,
        g.total_amount,
        g.passenger_count,
        g.trip_distance
    FROM read_parquet('../taxi_rides_ny/data/green/green_tripdata_2020-*.parquet') g
    LEFT JOIN read_csv('../taxi_rides_ny/seeds/taxi_zone_lookup.csv') z
        ON g.PULocationID = z.LocationID
    WHERE g.lpep_pickup_datetime >= '2019-01-01' 
      AND g.lpep_pickup_datetime < '2021-01-01'
),
all_trips AS (
    SELECT * FROM yellow_trips
    UNION ALL
    SELECT * FROM green_trips
),
monthly_zone_revenue AS (
    SELECT
        COALESCE(pickup_zone, 'Unknown Zone') as pickup_zone,
        revenue_month,
        service_type,
        SUM(fare_amount) as revenue_monthly_fare,
        SUM(extra) as revenue_monthly_extra,
        SUM(mta_tax) as revenue_monthly_mta_tax,
        SUM(tip_amount) as revenue_monthly_tip_amount,
        SUM(tolls_amount) as revenue_monthly_tolls_amount,
        SUM(ehail_fee) as revenue_monthly_ehail_fee,
        SUM(improvement_surcharge) as revenue_monthly_improvement_surcharge,
        SUM(total_amount) as revenue_monthly_total_amount,
        COUNT(*) as total_monthly_trips,
        AVG(passenger_count) as avg_monthly_passenger_count,
        AVG(trip_distance) as avg_monthly_trip_distance
    FROM all_trips
    GROUP BY pickup_zone, revenue_month, service_type
)
SELECT COUNT(*) as record_count
FROM monthly_zone_revenue
""").fetchone()

print(f"Total records in fct_monthly_zone_revenue: {result[0]:,}")

Total records in fct_monthly_zone_revenue: 12,248


In [4]:
result = conn.execute("""
SELECT COUNT(*) AS record_count
FROM dev.fct_monthly_zone_revenue
""").fetchone()

print(f"Total records: {result[0]}")

Total records: 511


### Question 4. Best Performing Zone for Green Taxis (2020)

In [ ]:
# Using the `fct_monthly_zone_revenue` table, find the pickup zone with 
# the **highest total revenue** (`revenue_monthly_total_amount`) for 
# **Green** taxi trips in 2020.

# Query directly from parquet files for all of 2020
# result = conn.execute("""
#     SELECT 
#         z.Zone as pickup_zone,
#         SUM(g.total_amount) as total_revenue
#     FROM read_parquet('../taxi_rides_ny/data/green/green_tripdata_2020-*.parquet') g
#     JOIN read_csv('../taxi_rides_ny/seeds/taxi_zone_lookup.csv') z
#         ON g.PULocationID = z.LocationID
#     WHERE EXTRACT(YEAR FROM g.lpep_pickup_datetime) = 2020
#     GROUP BY z.Zone
#     ORDER BY total_revenue DESC
#     LIMIT 1
# """).df()

result = conn.execute("""
    SELECT 
        pickup_zone,
        SUM(revenue_monthly_total_amount) as total_revenue
    FROM fct_monthly_zone_revenue
    WHERE service_type = 'Green'
        AND EXTRACT(YEAR FROM revenue_month) = 2020
    GROUP BY pickup_zone
    ORDER BY total_revenue DESC
    LIMIT 1
""").df()

result

,pickup_zone,total_revenue
0,East Harlem North,2069492.8


### Question 5. Green Taxi Trip Counts (October 2019)


In [100]:
# Using the `fct_monthly_zone_revenue` table, what is the **total number of trips** (`total_monthly_trips`) for Green taxis in October 2019?
result = conn.execute("""
    SELECT SUM(total_monthly_trips) AS total_trips
    FROM dev.fct_monthly_zone_revenue
    WHERE service_type = 'Green'
        AND EXTRACT(YEAR FROM revenue_month) = 2019
        AND EXTRACT(MONTH FROM revenue_month) = 10       
    
""").fetchone()

print(f"Total trips: {result[0]}")

Total trips: 384624


### 6. What is the count of records in `stg_fhv_tripdata`?

In [101]:
result = conn.execute("""
    SELECT COUNT(*) as record_count
    FROM dev.stg_fhv_tripdata
""").fetchone()
print(f"Total records: {result[0]:,}")

Total records: 43,244,693


In [102]:
conn.close()